In [8]:
import pandas as pd

import ast

In [2]:
DATE = '2024-08-16'
eval_results_df = pd.read_csv(f"s3://open-jobs-lake/job_quality/outputs/evaluation/JQ_evaluation_results_{DATE}.csv")
jq_error_analysis_df = pd.read_csv(f"s3://open-jobs-lake/job_quality/outputs/evaluation/JQ_prediction_errors_{DATE}.csv")

eval_results_df['True'] = eval_results_df['True'].apply(lambda x: ast.literal_eval(x))

## Show overall metrics per JQ measure

In [22]:
metrics = []
for i, row in eval_results_df.iterrows():
    m = {'jq_measure': row['JQ_measure_name']}
    m.update({k:round(v,3) for k,v in row['True'].items()})
    metrics.append(m)
metrics = pd.DataFrame(metrics)
metrics

,jq_measure,precision,recall,f1-score,support
0,L&D,0.953,0.854,0.901,48.0
1,CAREER,0.697,0.920,0.793,25.0
2,HOURS,0.855,1.000,0.922,59.0
3,FLEX_HOURS,0.609,0.800,0.691,35.0
4,SHIFT,0.800,0.235,0.364,17.0
5,LOC,0.500,0.250,0.333,36.0
6,FLEX_LOC,0.889,0.286,0.432,28.0
7,CONTRACT,0.909,0.513,0.656,39.0
8,LEAVE,0.652,1.000,0.789,30.0
9,COMP,1.000,0.849,0.918,86.0


In [20]:
for i, row in eval_results_df.iterrows():
    print(f"|{row['JQ_measure_name']}|{round(row['True']['support'])}|")

|L&D|48|
|CAREER|25|
|HOURS|59|
|FLEX_HOURS|35|
|SHIFT|17|
|LOC|36|
|FLEX_LOC|28|
|CONTRACT|39|
|LEAVE|30|
|COMP|86|
|PERKS|55|
|CARING|8|
|DISABILITY|2|
|HEALTH|7|
|M_HEALTH|4|
|SPONSORSHIP|2|
|REWARD|3|
|MISC|13|
|AUTONOMY|1|
|SENSE OF PURPOSE|4|
|SOCIAL|22|
|VOICE REPRESENTATION|0|


## Deeper dive
1. Examples of when it does badly for each JQ measure
2. Parent sectors that are better and worse

In [37]:
jq_measure = 'L&D'
error_type = 'FN' # 'FN', 'FP'
for i, row in jq_error_analysis_df[jq_error_analysis_df[jq_measure]==error_type].iterrows():
    print('---')
    print({k:v for k,v in ast.literal_eval(row['jq_sentences']).items() if jq_measure in v})
    print([v for v in ast.literal_eval(row['pred_ngram_matched']) if v[3] == jq_measure] )

---
{"At Tradewind you will have access to 25 fully certified CPD courses, that's 18 more than our next nearest competitor,  all focused on making you the best you can be.": ['L&D'], 'We care about your training and development more than any other agency - which is why we can offer you more certified CPD courses than any other education recruitment agency, 25 to be exact!': ['L&D']}
[]
---
{'· Can confidently read Engineering drawings Keywords  CNC, Miller, Milling, Machinist, Lathe, Turning, Setting, Operating, 3-axis, axis, Mil, Contract, Temporary, Permanent, Engineering, Technical, Technician, CNC, Machining, Machinist, Setting, Setter, Operating, Operator, Manufacturing, Production,  Cambridgeshire, Miller, Milling, Double Days, progression, training, Milling.': ['L&D']}
[]
---
{'You should be able to demonstrate ambition to succeed and will in turn be provided with the tools and training to fulfil that desire.': ['L&D']}
[]
---
{'An understanding of the basic principles of Revenu

In [17]:
jq_error_analysis_df.groupby('parent_sector')['n_incorrect'].mean()

parent_sector
Accountancy                    1.666667
Accountancy (Qualified)        2.833333
Admin, Secretarial &amp; PA    1.714286
Banking                        5.000000
Charity &amp; Voluntary        4.000000
Construction &amp; Property    3.000000
Education                      3.916667
Energy                         5.000000
Engineering                    4.500000
Estate Agency                  2.000000
FMCG                           1.000000
Health &amp; Medicine          3.250000
Hospitality &amp; Catering     2.777778
Human Resources                1.500000
IT &amp; Telecoms              1.333333
Marketing &amp; PR             2.666667
Motoring &amp; Automotive      3.500000
Recruitment Consultancy        4.000000
Retail                         2.250000
Sales                          3.000000
Social Care                    3.800000
Transport &amp; Logistics      2.285714
Name: n_incorrect, dtype: float64